# 04 m9_hybrid v1b Precision Gate Search

This notebook investigates the next m9 improvement: keep the v1 candidate scorer, but add simple deterministic acceptance gates after XGBoost scoring to reduce false positives.

**Important.** This notebook does not retrain m9. It reads the full-run outputs from `03_m9_hybrid_development.ipynb`, tunes gates on Alpha out-of-fold predictions only, and then evaluates Beta once.

**Goal.** Improve precision without destroying day-level F1. The default selection rule is: choose the best Alpha day F1 among gates with Alpha day precision at least `0.90`; if no gate meets that floor, choose best Alpha day F1 with precision as tie-breaker.

**Outputs.** All artifacts are written under `notebooks/99_Misc/outputs/04_m9_v1b_precision_gate_search/` and are ignored by git.

**Terms.** Reverse power flow (RPF) is real power flowing from the distribution network back into the transmission system.


## 1. Imports, Paths, And Controls

The controls below define the gate grid. No model training happens here; the notebook only filters already-scored candidate windows.

In [ ]:
from __future__ import annotations

import json
import time
from itertools import product
from pathlib import Path
from typing import Any, Iterable

import matplotlib
import numpy as np
import pandas as pd
import yaml

matplotlib.use("Agg")
import matplotlib.pyplot as plt

PALETTE = {
    "orange": "#eb932c",
    "dark_blue": "#22303d",
    "grey": "#2F4D67",
    "light_grey": "#5C7D99",
    "light_white": "#ebe3e3",
}
plt.rcParams.update({
    "font.family": "Arial",
    "axes.edgecolor": PALETTE["dark_blue"],
    "axes.labelcolor": PALETTE["dark_blue"],
    "axes.titlecolor": PALETTE["dark_blue"],
    "xtick.color": PALETTE["dark_blue"],
    "ytick.color": PALETTE["dark_blue"],
})

SEARCH_START_HOUR = 6
SEARCH_END_HOUR = 18
ALPHA_PRECISION_FLOOR = 0.90
THRESHOLD_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]
SOLAR_P95_GRID = [0.0, 2.0, 4.0]
SAME_SIGN_GRID = [0.0, 0.40, 0.60]
CORR_GRID = [-1.0, 0.0, 0.20]
SHAPE_GATE_GRID = [
    ("none", 0.0, 0.0),
    ("solar50", 0.50, 0.0),
    ("net50", 0.0, 0.50),
    ("both50", 0.50, 0.50),
    ("both70", 0.70, 0.70),
]
MAX_DURATION_GRID = [8.0, 6.0]


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "publication" / "2_journal_article" / "config" / "experiment_config.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not find PyNRPF repo root.")


REPO_ROOT = find_repo_root()
ARTICLE_ROOT = REPO_ROOT / "publication" / "2_journal_article"
MISC_DIR = ARTICLE_ROOT / "notebooks" / "99_Misc"
M9_OUTPUT = MISC_DIR / "outputs" / "03_m9_hybrid_development"
OUTPUT_ROOT = MISC_DIR / "outputs" / "04_m9_v1b_precision_gate_search"
INTERMEDIATE_DIR = OUTPUT_ROOT / "intermediate"
METRICS_DIR = OUTPUT_ROOT / "metrics"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [INTERMEDIATE_DIR, METRICS_DIR, FIGURES_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

with (ARTICLE_ROOT / "config" / "experiment_config.yaml").open("r", encoding="utf-8") as fh:
    CFG = yaml.safe_load(fh)

print("M9 source output:", M9_OUTPUT)
print("Gate output:", OUTPUT_ROOT)


## 2. Load Existing m9 Full-Run Artifacts

This notebook expects the full m9 run to have produced Alpha out-of-fold scored candidates and Beta transfer scored candidates. If these files are missing, rerun Notebook 03 with `RUN_FULL_M9=True`.

In [ ]:
EXPECTED_COLUMNS = [
    "substation_id", "date", "timestamp", "net_load_MW", "solar_MW", "label_interval", "label_day"
]
COUNT_COLUMNS = ["support", "positive_support", "tp", "fp", "fn", "tn"]
SCORE_COLUMNS = ["precision", "recall", "f1"]


def load_final_dataset(dataset_key: str) -> pd.DataFrame:
    path = ARTICLE_ROOT / CFG["paths"][f"{dataset_key}_dataset_path"]
    df = pd.read_parquet(path)[EXPECTED_COLUMNS].copy()
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    if getattr(ts.dt, "tz", None) is not None:
        ts = ts.dt.tz_localize(None)
    df["timestamp"] = ts
    df["date"] = df["date"].astype(str)
    df["label_interval"] = df["label_interval"].astype(bool)
    df["label_day"] = df.groupby(["substation_id", "date"])["label_interval"].transform("any")
    return df.sort_values(["substation_id", "timestamp"]).reset_index(drop=True)


def read_required_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Required m9 full-run artifact not found: {path}")
    df = pd.read_csv(path)
    for col in ["pred_start", "pred_end", "true_start", "true_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    df["date"] = df["date"].astype(str)
    return df


alpha = load_final_dataset("alpha")
beta = load_final_dataset("beta")
alpha_scored = read_required_csv(M9_OUTPUT / "intermediate" / "04_alpha_loso_scored_candidates.csv")
beta_scored = read_required_csv(M9_OUTPUT / "intermediate" / "06_beta_scored_candidates.csv")

print(f"Alpha scored candidates: {len(alpha_scored):,}")
print(f"Beta scored candidates: {len(beta_scored):,}")


## 3. Gate Evaluation Helpers

A gate filters candidate windows before the one-window decoder. If no candidate passes the gate for a site-day, that day is decoded as no RPF.

In [ ]:
def write_csv(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path


def binary_metrics(y_true: Iterable[Any], y_pred: Iterable[Any]) -> dict[str, Any]:
    true = np.asarray(list(y_true), dtype=bool)
    pred = np.asarray(list(y_pred), dtype=bool)
    tp = int((true & pred).sum())
    fp = int((~true & pred).sum())
    fn = int((true & ~pred).sum())
    tn = int((~true & ~pred).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"support": len(true), "positive_support": int(true.sum()), "tp": tp, "fp": fp, "fn": fn, "tn": tn, "precision": precision, "recall": recall, "f1": f1}


def true_windows(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (site, date), grp in df.groupby(["substation_id", "date"], sort=False):
        labelled = grp.loc[grp["label_interval"]]
        rows.append({
            "substation_id": site,
            "date": date,
            "label_day": not labelled.empty,
            "true_start": labelled["timestamp"].iloc[0] if not labelled.empty else pd.NaT,
            "true_end": labelled["timestamp"].iloc[-1] if not labelled.empty else pd.NaT,
        })
    return pd.DataFrame(rows)


def gate_mask(scored: pd.DataFrame, gate: dict[str, Any]) -> pd.Series:
    return (
        (scored["candidate_probability"] >= gate["threshold"])
        & (scored["solar_p95_inside"] >= gate["min_solar_p95"])
        & (scored["derivative_same_sign_fraction"] >= gate["min_same_sign"])
        & (scored["solar_net_corr"] >= gate["min_corr"])
        & (scored["solar_bell_score"] >= gate["min_solar_bell"])
        & (scored["net_load_n_shape_score"] >= gate["min_net_shape"])
        & (scored["duration_hours"] <= gate["max_duration_hours"])
    )


def select_best_candidates(scored: pd.DataFrame, gate: dict[str, Any]) -> pd.DataFrame:
    passed = scored.loc[gate_mask(scored, gate)].copy()
    if passed.empty:
        return pd.DataFrame(columns=["substation_id", "date", "candidate_id", "pred_start", "pred_end", "candidate_probability"])
    return (
        passed.sort_values(["substation_id", "date", "candidate_probability"], ascending=[True, True, False])
        .groupby(["substation_id", "date"], as_index=False)
        .head(1)
    )


def decode_days_only(days: pd.DataFrame, scored: pd.DataFrame, gate: dict[str, Any]) -> pd.DataFrame:
    best = select_best_candidates(scored, gate)
    keep_cols = ["substation_id", "date", "candidate_id", "pred_start", "pred_end", "candidate_probability"]
    for feature in ["solar_p95_inside", "solar_bell_score", "net_load_n_shape_score", "derivative_same_sign_fraction", "solar_net_corr", "duration_hours"]:
        if feature in best.columns:
            keep_cols.append(feature)
    decoded = days.merge(best[keep_cols], on=["substation_id", "date"], how="left")
    decoded["candidate_probability"] = decoded["candidate_probability"].fillna(0.0)
    decoded["pred_day"] = decoded["pred_start"].notna()
    return decoded


def decode_days(eval_df: pd.DataFrame, scored: pd.DataFrame, gate: dict[str, Any], days: pd.DataFrame | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    day_labels = true_windows(eval_df) if days is None else days.copy()
    decoded = decode_days_only(day_labels, scored, gate)
    interval_frame = eval_df.merge(decoded[["substation_id", "date", "pred_day", "pred_start", "pred_end"]], on=["substation_id", "date"], how="left")
    interval_frame["pred_interval"] = (
        interval_frame["pred_day"].fillna(False)
        & (interval_frame["timestamp"] >= interval_frame["pred_start"])
        & (interval_frame["timestamp"] <= interval_frame["pred_end"])
    )
    interval_frame["hour"] = interval_frame["timestamp"].dt.hour
    return decoded, interval_frame


def day_metrics(days: pd.DataFrame, scored: pd.DataFrame, gate: dict[str, Any]) -> dict[str, Any]:
    decoded = decode_days_only(days, scored, gate)
    return binary_metrics(decoded["label_day"], decoded["pred_day"])


def all_metrics(eval_df: pd.DataFrame, scored: pd.DataFrame, dataset: str, fold_id: str, gate: dict[str, Any]) -> pd.DataFrame:
    decoded, interval_frame = decode_days(eval_df, scored, gate)
    rows = [{"dataset": dataset, "fold_id": fold_id, "level": "day", **binary_metrics(decoded["label_day"], decoded["pred_day"])}]
    daytime = interval_frame["hour"].between(SEARCH_START_HOUR, SEARCH_END_HOUR, inclusive="both")
    rows.append({"dataset": dataset, "fold_id": fold_id, "level": "interval", **binary_metrics(interval_frame.loc[daytime, "label_interval"], interval_frame.loc[daytime, "pred_interval"])})
    return pd.DataFrame(rows)


def make_gate_grid() -> list[dict[str, Any]]:
    gates = []
    for threshold, min_solar_p95, min_same_sign, min_corr, (shape_id, min_solar_bell, min_net_shape), max_duration in product(
        THRESHOLD_GRID, SOLAR_P95_GRID, SAME_SIGN_GRID, CORR_GRID, SHAPE_GATE_GRID, MAX_DURATION_GRID
    ):
        gates.append({
            "gate_id": f"thr{threshold:.2f}_sol{min_solar_p95:g}_same{min_same_sign:.2f}_corr{min_corr:.2f}_{shape_id}_dur{max_duration:g}",
            "threshold": float(threshold),
            "min_solar_p95": float(min_solar_p95),
            "min_same_sign": float(min_same_sign),
            "min_corr": float(min_corr),
            "shape_gate": shape_id,
            "min_solar_bell": float(min_solar_bell),
            "min_net_shape": float(min_net_shape),
            "max_duration_hours": float(max_duration),
        })
    return gates


## 4. Alpha-Only Gate Search

This section searches gates using only Alpha out-of-fold scored candidates. Beta labels are not used for selection.

In [ ]:
t0 = time.perf_counter()
alpha_days = true_windows(alpha)
gate_rows = []
for gate in make_gate_grid():
    metrics = day_metrics(alpha_days, alpha_scored, gate)
    gate_rows.append({**gate, **{f"alpha_day_{k}": v for k, v in metrics.items()}})

gate_sweep = pd.DataFrame(gate_rows)
write_csv(gate_sweep, INTERMEDIATE_DIR / "01_alpha_gate_sweep.csv")

eligible = gate_sweep.loc[gate_sweep["alpha_day_precision"] >= ALPHA_PRECISION_FLOOR].copy()
selection_pool = eligible if not eligible.empty else gate_sweep
selection_reason = f"alpha_precision_floor_{ALPHA_PRECISION_FLOOR}" if not eligible.empty else "no_gate_met_precision_floor"
selected_row = selection_pool.sort_values(["alpha_day_f1", "alpha_day_precision", "alpha_day_recall"], ascending=[False, False, False]).iloc[0]
selected_gate = {col: selected_row[col] for col in ["gate_id", "threshold", "min_solar_p95", "min_same_sign", "min_corr", "shape_gate", "min_solar_bell", "min_net_shape", "max_duration_hours"]}

print("Selection reason:", selection_reason)
print("Selected gate:")
display(pd.DataFrame([selected_gate]))
print("Selected Alpha day metrics:")
display(selected_row[[col for col in selected_row.index if col.startswith("alpha_day_")]].to_frame().T)


## 5. Evaluate Selected Gate On Alpha And Beta

After selection, the same gate is applied to Beta once. This keeps the protocol clean: Alpha chooses, Beta evaluates.

In [ ]:
alpha_metrics = all_metrics(alpha, alpha_scored, dataset="Alpha", fold_id="alpha_oof_selected_gate", gate=selected_gate)
beta_metrics = all_metrics(beta, beta_scored, dataset="Beta", fold_id="beta_transfer_selected_gate", gate=selected_gate)
beta_site_metrics = pd.concat([
    all_metrics(
        beta.loc[beta["substation_id"].eq(site)],
        beta_scored.loc[beta_scored["substation_id"].eq(site)],
        dataset="Beta",
        fold_id=f"beta_site_{site}",
        gate=selected_gate,
    )
    for site in sorted(beta["substation_id"].unique())
], ignore_index=True)

beta_decoded, beta_interval_frame = decode_days(beta, beta_scored, selected_gate)
beta_decoded["confusion_group"] = np.select(
    [
        beta_decoded["label_day"] & beta_decoded["pred_day"],
        ~beta_decoded["label_day"] & ~beta_decoded["pred_day"],
        ~beta_decoded["label_day"] & beta_decoded["pred_day"],
        beta_decoded["label_day"] & ~beta_decoded["pred_day"],
    ],
    ["TP", "TN", "FP", "FN"],
    default="unknown",
)

write_csv(pd.DataFrame([selected_gate | {"selection_reason": selection_reason}]), METRICS_DIR / "01_selected_gate.csv")
write_csv(alpha_metrics, METRICS_DIR / "02_alpha_selected_gate_metrics.csv")
write_csv(beta_metrics, METRICS_DIR / "03_beta_selected_gate_metrics.csv")
write_csv(beta_site_metrics, METRICS_DIR / "04_beta_site_selected_gate_metrics.csv")
write_csv(beta_decoded, INTERMEDIATE_DIR / "02_beta_selected_gate_decoded_days.csv")

print("Alpha selected-gate metrics")
display(alpha_metrics)
print("Beta selected-gate metrics")
display(beta_metrics)


## 6. Feature Diagnostics And Figures

The first figure shows the Alpha gate-search frontier. The second and third figures inspect whether the selected gate reduced Beta false positives and where failures remain.

In [ ]:
def save_alpha_frontier(gate_sweep: pd.DataFrame, path: Path) -> Path:
    fig, ax = plt.subplots(figsize=(6.8, 4.8))
    scatter = ax.scatter(gate_sweep["alpha_day_precision"], gate_sweep["alpha_day_recall"], c=gate_sweep["alpha_day_f1"], cmap="viridis", s=18, alpha=0.75)
    ax.axvline(ALPHA_PRECISION_FLOOR, color=PALETTE["orange"], linestyle="--", linewidth=1.2, label="precision floor")
    ax.scatter([selected_row["alpha_day_precision"]], [selected_row["alpha_day_recall"]], color=PALETTE["orange"], edgecolor=PALETTE["dark_blue"], s=90, label="selected")
    ax.set_xlabel("Alpha day precision")
    ax.set_ylabel("Alpha day recall")
    ax.set_title("Alpha gate search frontier")
    ax.set_axisbelow(True)
    ax.grid(color=PALETTE["light_white"], linewidth=0.8)
    fig.colorbar(scatter, ax=ax, label="Alpha day F1")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path


def save_beta_site_bars(beta_site_metrics: pd.DataFrame, path: Path) -> Path:
    day = beta_site_metrics.loc[beta_site_metrics["level"].eq("day")].copy()
    day["site"] = day["fold_id"].str.replace("beta_site_", "", regex=False)
    day = day.sort_values("f1", ascending=False)
    x = np.arange(len(day))
    width = 0.25
    fig, ax = plt.subplots(figsize=(8.0, 4.2))
    ax.bar(x - width, day["precision"], width, label="Precision", color=PALETTE["dark_blue"])
    ax.bar(x, day["recall"], width, label="Recall", color=PALETTE["orange"])
    ax.bar(x + width, day["f1"], width, label="F1", color=PALETTE["grey"])
    ax.set_xticks(x)
    ax.set_xticklabels(day["site"], rotation=45)
    ax.set_ylim(0, 1)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=PALETTE["light_white"], linewidth=0.8)
    ax.set_title("Beta day metrics after selected precision gate")
    ax.legend(frameon=False, ncol=3)
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path


def save_beta_feature_boxplots(beta_decoded: pd.DataFrame, path: Path) -> Path:
    plot_features = ["candidate_probability", "solar_p95_inside", "derivative_same_sign_fraction", "solar_net_corr", "solar_bell_score", "net_load_n_shape_score"]
    groups = ["TP", "FP"]
    fig, axes = plt.subplots(2, 3, figsize=(10.5, 6.4))
    for ax, feature in zip(axes.ravel(), plot_features):
        data = [beta_decoded.loc[beta_decoded["confusion_group"].eq(group), feature].dropna().to_numpy() for group in groups]
        ax.boxplot(data, tick_labels=groups, patch_artist=True, boxprops=dict(facecolor=PALETTE["light_white"], color=PALETTE["dark_blue"]), medianprops=dict(color=PALETTE["orange"]))
        ax.set_title(feature, fontsize=10)
        ax.set_axisbelow(True)
        ax.grid(axis="y", color=PALETTE["light_white"], linewidth=0.8)
    fig.suptitle("Selected-candidate features: Beta TP vs FP days", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path


frontier_fig = save_alpha_frontier(gate_sweep, FIGURES_DIR / "fig01_alpha_gate_search_frontier.png")
site_fig = save_beta_site_bars(beta_site_metrics, FIGURES_DIR / "fig02_beta_site_day_metrics_selected_gate.png")
feature_fig = save_beta_feature_boxplots(beta_decoded, FIGURES_DIR / "fig03_beta_tp_fp_feature_boxplots.png")

feature_summary = (
    beta_decoded.groupby("confusion_group")[["candidate_probability", "solar_p95_inside", "derivative_same_sign_fraction", "solar_net_corr", "solar_bell_score", "net_load_n_shape_score", "duration_hours"]]
    .agg(["count", "mean", "median"])
)
feature_summary.columns = ["_".join(col).strip() for col in feature_summary.columns]
feature_summary = feature_summary.reset_index()
write_csv(feature_summary, INTERMEDIATE_DIR / "03_beta_confusion_feature_summary.csv")
display(feature_summary)


## 7. Manifest And Inventory

In [ ]:
manifest = {
    "mode": "m9_v1b_precision_gate_search",
    "elapsed_seconds": time.perf_counter() - t0,
    "alpha_precision_floor": ALPHA_PRECISION_FLOOR,
    "selection_reason": selection_reason,
    "selected_gate": selected_gate,
    "outputs": {
        "metrics": [
            "01_selected_gate.csv",
            "02_alpha_selected_gate_metrics.csv",
            "03_beta_selected_gate_metrics.csv",
            "04_beta_site_selected_gate_metrics.csv",
        ],
        "figures": [frontier_fig.name, site_fig.name, feature_fig.name],
    },
}
with (MANIFEST_DIR / "run_manifest.json").open("w", encoding="utf-8") as fh:
    json.dump(manifest, fh, indent=2, default=str)

for folder in [INTERMEDIATE_DIR, METRICS_DIR, FIGURES_DIR, MANIFEST_DIR]:
    print(f"\n{folder.relative_to(OUTPUT_ROOT)}")
    for item in sorted(folder.glob("*")):
        print(" -", item.name)
